Overall goal: sort a pile of images into two categories (binary classification) one with AI images, and one with real images

In [1]:
#imports
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import os
from PIL import Image


#Quick test to check hardware
#Check CPU cores, GPU availability, and  if it's compatible with PyTorch
print(f"CPU cores: {os.cpu_count()}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch version: {torch.__version__}")

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

x = torch.tensor([1.0, 2.0, 3.0]).to(device)
print(f"Test tensor device: {x.device}")


fulldataset = "Art_shuffled"


ModuleNotFoundError: No module named 'torch'

In [80]:
#We resize the images so they fit ResNet18 (224x224) 
#and transform for more variety

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [81]:
#Loading the dataset and splitting for training

full_dataset = datasets.ImageFolder(root="Art_shuffled", transform=train_transforms)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

val_dataset.dataset.transform = val_transforms

#Useble hyperparameters will vary based on hardware
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          num_workers=10, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False,
                        num_workers=10, pin_memory=True)

print(full_dataset.class_to_idx)



{'AiArtData': 0, 'RealArt': 1}


In [71]:
#We select our model and its pretrained weights
model = models.resnet18(weights="IMAGENET1K_V1")

for param in model.parameters():
    param.requires_grad = False

model.fc =nn.Sequential(
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 1),
    nn.Sigmoid()
)

model = model.to(device)


In [72]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.float().to(device)
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += ((outputs > 0.5) == labels).sum().item()
    
    return total_loss / len(loader), correct / len(loader.dataset)

def val_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.float().to(device)
            output = model(images).squeeze()
            loss = criterion(output, labels)
            total_loss += loss.item()
            correct += ((output > 0.5) == labels).sum().item()
        
    return total_loss / len(loader), correct / len(loader.dataset)
    
for epoch in range (5):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    va_loss, va_acc = val_epoch(model, val_loader, criterion)

    print(f"Epoch {epoch+1}: Train Acc={tr_acc:.3f}, Val Acc={va_acc:.3f}")
    

Epoch 1: Train Acc=0.611, Val Acc=0.686
Epoch 2: Train Acc=0.737, Val Acc=0.613
Epoch 3: Train Acc=0.749, Val Acc=0.780
Epoch 4: Train Acc=0.772, Val Acc=0.754
Epoch 5: Train Acc=0.782, Val Acc=0.754


In [73]:
for param in model.layer4.parameters():
    param.requires_grad = True

optimizer2 = torch.optim.Adam([
    {"params": model.layer4.parameters(), "lr": 1e-5},
    {"params": model.fc.parameters(), "lr": 1e-4}
])

for epoch  in range(5):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer2, criterion)
    va_loss, va_acc = val_epoch(model, val_loader, criterion)
    
    print(f"Fine tune Epoch {epoch+1}: Train Acc={tr_acc:.3f}, Val Acc={va_acc:.3f}")
    

Fine tune Epoch 1: Train Acc=0.816, Val Acc=0.791
Fine tune Epoch 2: Train Acc=0.859, Val Acc=0.791
Fine tune Epoch 3: Train Acc=0.900, Val Acc=0.796
Fine tune Epoch 4: Train Acc=0.918, Val Acc=0.801
Fine tune Epoch 5: Train Acc=0.934, Val Acc=0.796


In [74]:
#We save the model for later use
torch.save(model.state_dict(), "ai_detector_ResNet18.pth")

#We reload the model to test it on new images without retraining
model2 = models.resnet18(weights=None)
model2.fc = nn.Sequential(
    nn.Linear(model2.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 1),
    nn.Sigmoid()
)

model2.load_state_dict(torch.load("ai_detector_ResNet18.pth"))
model2 = model2.to(device)
model2.eval()


def classify_folder(folder_path, model, transform, device):
    model.eval()
    results = []
    for true_label in ["AiArtData", "RealArt"]:
        subfolder = os.path.join(folder_path, true_label)
        
        for filename in os.listdir(subfolder):
            image_path = os.path.join(subfolder, filename)
            img = Image.open(image_path).convert("RGB")
            img_tensor = transform(img).unsqueeze(0).to(device)

            with torch.no_grad():
                prod = model(img_tensor).item()

            pred_label = "AiArtData" if prod > 0.5 else "RealArt"
            confidence = prod if prod > 0.5 else 1 - prod
            correct = pred_label == true_label
            results.append({
                "file": filename,
                "true": true_label,
                "predicted": pred_label,
                "confidence": confidence,
                "correct": correct
            })

            status = "CORRECT" if correct else "WRONG"
            print(f"{status} - File: {filename}")
            print(f"___________True label: {true_label}")
            print(f"___________Predicted: {pred_label}")
            print(f"___________Confidence: {confidence:.1%}\n")
        
        
    total = len(results)
    correct_count = sum(r["correct"] for r in results)
    print(f"Summary: Correct: {correct_count}/{total} ({correct_count/total:.1%})")
    return results    

results = classify_folder(
    folder_path="task2_new_images",
    model=model2,
    transform=val_transforms,
    device=device
)


WRONG - File: Screenshot 2026-04-22 at 13.21.45.png
___________True label: AiArtData
___________Predicted: RealArt
___________Confidence: 88.0%

CORRECT - File: Screenshot 2026-04-22 at 13.21.56.png
___________True label: AiArtData
___________Predicted: AiArtData
___________Confidence: 92.4%

CORRECT - File: Screenshot 2026-04-22 at 15.23.13.png
___________True label: AiArtData
___________Predicted: AiArtData
___________Confidence: 78.0%

CORRECT - File: IMG_8190.jpeg
___________True label: RealArt
___________Predicted: RealArt
___________Confidence: 58.7%

CORRECT - File: IMG_8191.jpeg
___________True label: RealArt
___________Predicted: RealArt
___________Confidence: 92.9%

Summary: Correct: 4/5 (80.0%)
